# Fetch the Llama-3.1-8B prefix the Llama member needs

`alexchilton/ntl-ts-llama31-8b` documents its base in `meta.json`: `model_dir: meta-llama/Llama-3.1-8B`,
`keep: 24`, `layers: [2,4,6,8,12,16,20,24]`, `max_len: 447`. Kaggle's copy of Llama 3.1 sits behind Meta's
access form, so this kernel takes an ungated Hugging Face mirror instead and keeps only what the member
reads: the embedding, the first 24 decoder layers, the final norm and the tokenizer. That is about 12 GB
in bf16 instead of 16, and it attaches to the training kernel as an ordinary dataset.

Output: `/kaggle/working/llama31-prefix24/` with `config.json` (num_hidden_layers = 24),
`model.safetensors`, and the tokenizer.

In [ ]:
"""Fetch Llama-3.1-8B weights from an ungated Hugging Face mirror and trim to the 24 layers the member needs.

The member `alexchilton/ntl-ts-llama31-8b` documents its base in meta.json: model_dir meta-llama/Llama-3.1-8B,
keep 24, layers [2,4,6,8,12,16,20,24], max_len 447. Kaggle's copy of Llama 3.1 is behind Meta's access form,
so this kernel downloads an ungated mirror instead, keeps the first 24 decoder layers plus the embedding and
norm (that is all the member reads), and writes them as one dataset the training kernel can attach.
"""
import json
import os
import time

T0 = time.time()
MIRROR = "NousResearch/Meta-Llama-3.1-8B"
OUT = "/kaggle/working/llama31-prefix24"
KEEP = 24
os.makedirs(OUT, exist_ok=True)
print(f"mirror: {MIRROR} | keep {KEEP} layers | out {OUT}", flush=True)

## Download the mirror

In [ ]:
from huggingface_hub import snapshot_download

# config + tokenizer + the index are small; the weights are sharded
src = snapshot_download(
    repo_id=MIRROR,
    allow_patterns=["*.json", "*.model", "tokenizer*", "*.txt", "*.safetensors"],
    local_dir="/kaggle/working/src",
)
print("downloaded to", src, flush=True)
for root, _, files in os.walk(src):
    for f in sorted(files)[:40]:
        p = os.path.join(root, f)
        print(f"  {os.path.relpath(p, src):<50} {os.path.getsize(p):>14,}", flush=True)

## Trim to 24 layers

In [ ]:
import glob
import json as _json
import shutil

import torch
from safetensors import safe_open
from safetensors.torch import save_file

cfg = _json.load(open(f"{src}/config.json"))
n_layers = cfg["num_hidden_layers"]
print(f"source layers {n_layers} -> keeping {KEEP}", flush=True)
cfg["num_hidden_layers"] = KEEP
for key in ("layer_types",):
    if cfg.get(key):
        cfg[key] = cfg[key][:KEEP]
_json.dump(cfg, open(f"{OUT}/config.json", "w"), indent=1)

keep_prefixes = ("model.embed_tokens.", "model.norm.", "lm_head.")
out_tensors, dropped, kept = {}, 0, 0
for shard in sorted(glob.glob(f"{src}/*.safetensors")):
    with safe_open(shard, framework="pt") as f:
        for name in f.keys():
            if any(name.startswith(p) for p in keep_prefixes):
                out_tensors[name] = f.get_tensor(name)
                kept += 1
                continue
            if name.startswith("model.layers."):
                idx = int(name.split(".")[2])
                if idx < KEEP:
                    out_tensors[name] = f.get_tensor(name)
                    kept += 1
                else:
                    dropped += 1
            else:
                dropped += 1
print(f"kept {kept} tensors, dropped {dropped}", flush=True)
save_file(out_tensors, f"{OUT}/model.safetensors")
print(f"wrote {OUT}/model.safetensors  {os.path.getsize(OUT + '/model.safetensors'):,} bytes", flush=True)

for pat in ("tokenizer*", "special_tokens_map.json", "*.model", "generation_config.json"):
    for p in glob.glob(f"{src}/{pat}"):
        shutil.copy(p, OUT)
print("files in output:", sorted(os.listdir(OUT)), flush=True)
print(f"done in {time.time() - T0:.0f}s", flush=True)